## Example 2 - 간단한 신경망을 직접 만들고, 역전파(Backpropagation)로 학습

Copyright 2024, Joshua Starmer, 편집 및 수정: YoungDon Choi (K-water Research Institute)

### 노트북 개요

> **📌 이 Notebook에서 다룰 내용은**
>
> - 먼저 아래 첫 번째 이미지처럼 **학습되지 않은(untrained)** 무작위 가중치·편향으로 신경망을 시작합니다.
> - 그 다음 **역전파(Backpropagation)** 알고리즘으로 학습을 진행해서, 두 번째 이미지처럼 1장에서 봤던 **학습이 완료된(trained)** 가중치·편향 값에 최대한 가깝게 만드는 것이 목표입니다.
>
> 1. **[학습되지 않은 가중치·편향을 가진 신경망 코딩하기](#untrained)** — 이번에는 `nn.Module` 대신 `LightningModule`을 상속받는 클래스의 기본 구조를 보여줍니다.
> 2. **[신경망의 가중치와 편향 학습시키기](#train)** — 간단한 신경망의 가중치·편향을 실제로 학습(훈련)시키는 방법을 보여줍니다.

<img src="./images/chapter_2_train_all.png" alt="an untrained neural network" style="width: 800px;">

<img src="./images/chapter_1_pre_trained_nn_labeled.png" alt="a trained neural network" style="width: 800px;">

### 신경망을 만들고 학습시키는 데 필요한 파이썬 모듈(module, 라이브러리)들을 불러오는(import) 단계

> 이번 노트북에서 사용할 라이브러리를 불러오는 코드입니다. 
> - `import torch`, `import torch.nn as nn`, `import torch.nn.functional as F` — Example-1과 동일하게 텐서 생성, 신경망 클래스 구성 요소, ReLU 같은 활성화 함수를 제공합니다.
> - `from torch.optim import SGD` — **확률적 경사 하강법(Stochastic Gradient Descent)** 최적화 알고리즘입니다. 이름은 "확률적(Stochastic)"이지만, 이 노트북에서는 매번 데이터의 일부(랜덤 부분집합)만 뽑아 쓰는 대신 **전체 학습 데이터를 한 번에** 통과시키므로, 실질적으로는 그냥 "경사 하강법(Gradient Descent)"처럼 동작합니다.
> - `import lightning as L` — **PyTorch Lightning** 라이브러리입니다. 학습 코드를 더 간결하게 작성하고, 최적화·확장을 쉽게 해줍니다.
> - `from torch.utils.data import TensorDataset, DataLoader` — 학습 데이터를 담아두는 `DataLoader`를 만들기 위한 도구입니다.
> - `import matplotlib.pyplot as plt`, `import seaborn as sns` — 그래프를 그리기 위한 라이브러리입니다.

In [ ]:
import torch ## torch let's us create tensors and also provides helper functions
import torch.nn as nn ## torch.nn gives us nn.Module(), nn.Embedding() and nn.Linear()
import torch.nn.functional as F # This gives us relu()
from torch.optim import SGD 
import lightning as L ## Lightning makes it easier to write, optimize and scale our code
from torch.utils.data import TensorDataset, DataLoader ## We'll store our data in DataLoaders

import matplotlib.pyplot as plt ## matplotlib allows us to draw graphs.
import seaborn as sns ## seaborn makes it easier to draw nice-looking graphs.

### 훈련데이터셋 만들기

간단한 학습 데이터셋을 소개합니다. 아래 이미지처럼 데이터는 딱 **3개의 점**으로 이루어져 있습니다(투여량 0.0, 0.5, 1.0에 대한 약효).
>
> 데이터를 꼭 `DataLoader`에 담을 필요는 없지만, `DataLoader`를 쓰면 여러 편리한 기능을 얻을 수 있습니다.
> - 데이터가 아주 많을 때, 한꺼번에 메모리(RAM)에 다 올리지 않고 **배치(batch) 단위**로 나눠서 쉽게 가져올 수 있습니다.
> - 매 에폭(epoch)마다 데이터를 자동으로 **셔플(섞기)** 해줄 수 있습니다.
> - 디버깅 목적으로 데이터의 일부만 빠르게 써보고 싶을 때도 편리합니다.

<img src="images/chapter_2_training_data.png" alt="a simple dataset for training" style="width: 800px;">

> 학습 데이터를 실제로 만드는 코드입니다.
>
> - `training_inputs = torch.tensor([0.0, 0.5, 1.0])` — x축 값(투여량, dose)들입니다.
> - `training_labels = torch.tensor([0.0, 1.0, 0.0])` — 각 투여량에 대응하는 y축 값(약의 효과, effectiveness)들입니다. 즉 정답(label) 값입니다.
> - `training_dataset = TensorDataset(training_inputs, training_labels)` — 입력과 정답 텐서를 하나의 데이터셋 객체로 묶습니다.
> - `dataloader = DataLoader(training_dataset)` — 이 데이터셋을 `DataLoader`로 감싸서, 이후 학습(training) 과정에서 쉽게 꺼내 쓸 수 있도록 준비합니다.

In [ ]:
training_inputs = torch.tensor([0.0, 0.5, 1.0])

training_labels = torch.tensor([0.0, 1.0, 0.0])

training_dataset = TensorDataset(training_inputs, training_labels)
dataloader = DataLoader(training_dataset)

### **학습 가능한(trainable)** 가중치(Weights)와 편향(Biases)을 가진 신경망을 만들기

Example-1에서는 `nn.Module`을 상속받았지만, 이번에는 **`L.LightningModule`**을 상속받습니다. `LightningModule`은 `nn.Module`이 제공하는 모든 기능에 더해, 사용할 **최적화 알고리즘(optimizer)** 을 정의하고, 학습의 매 스텝(training step)마다 어떤 계산을 할지도 함께 정의할 수 있게 해줍니다.

> 학습 가능한 신경망 클래스 `myNN`을 정의하는 핵심 코드입니다. `L.LightningModule`을 상속받으며, Example-1의 `myNN`보다 메서드가 두 개 더 많습니다.
>
> **1) `__init__()` — 가중치·편향을 무작위 값으로 초기화**
> - Example-1과 달리 이번에는 가중치·편향을 **무작위(random) 값**으로 초기화합니다(예: `w1 = 0.06`, `b1 = 0.0` 등). 이것이 "학습되지 않은(untrained)" 상태입니다.
> - 이번에는 `torch.tensor()`를 그냥 쓰지 않고, `nn.Parameter()`로 한 번 더 감쌉니다. PyTorch는 **`nn.Parameter()`로 감싼 값만 학습(최적화) 대상으로 인식**하기 때문입니다. `nn.Parameter()`는 파라미터를 만드는 가장 기본적인 방법이며, 이후 다른 노트북에서 더 다양한 방법을 보게 됩니다.
> - `self.loss = nn.MSELoss(reduction='sum')` — **평균제곱오차(Mean Squared Error)** 손실 함수를 정의합니다. `reduction='sum'`이므로 각 데이터 포인트의 오차 제곱을 평균이 아니라 **합산**해서 손실값을 계산합니다.
>
> **2) `forward()` — Example=1과 완전히 동일한 순전파 계산**
> - 입력값에 w1·b1을 적용하고 ReLU를 통과시킨 값(위쪽 경로)과, w2·b2를 적용하고 ReLU를 통과시킨 값(아래쪽 경로)을 각각 w3, w4로 스케일한 뒤 더해서 최종 출력을 계산합니다. 계산 로직 자체는 Example-1 신경망과 완전히 같습니다.
>
> **3) `configure_optimizers()` — 사용할 최적화 알고리즘 설정**
> - `SGD(self.parameters(), lr=0.01)`을 반환합니다. `self.parameters()`는 `nn.Parameter()`로 만들어진 모든 학습 대상(w1~w4, b1, b2)을 가져오고, `lr=0.01`은 학습률(learning rate, 한 번에 얼마나 크게 값을 업데이트할지)입니다.
> - 주석에서 설명하듯 PyTorch에는 순수한 "경사 하강법(Gradient Descent)" 옵티마이저가 따로 없고 `SGD`만 있습니다. 하지만 매번 3개의 투여량 데이터를 전부(랜덤 일부가 아니라) 신경망에 통과시키기 때문에, 사실상 일반적인 경사 하강법처럼 동작합니다.
>
> **4) `training_step()` — 학습 한 스텝에서 할 일 정의**
> - `LightningModule`이 학습 도중 매 스텝마다 자동으로 호출하는 메서드입니다.
> - `inputs, labels = batch` — `DataLoader`에서 꺼낸 배치(batch)에서 입력값과 정답값을 분리합니다.
> - `outputs = self.forward(inputs)` — 입력을 신경망에 통과시켜 예측값을 얻습니다.
> - `loss = self.loss(outputs, labels)` — 예측값(outputs)과 실제 정답(labels)의 차이를 손실(loss)로 계산합니다. 이 손실값이 작을수록 신경망이 데이터를 잘 예측한다는 뜻입니다.
> - 계산된 `loss`를 반환하면, Lightning이 내부적으로 이 값을 이용해 역전파(backpropagation)를 수행하고 가중치·편향을 업데이트합니다.

In [ ]:
class myNN(L.LightningModule):

    def __init__(self):

        super().__init__()

        self.w1 = nn.Parameter(torch.tensor(0.06))
        self.b1 = nn.Parameter(torch.tensor(0.0))

        self.w2 = nn.Parameter(torch.tensor(3.49))
        self.b2 = nn.Parameter(torch.tensor(0.0))

        self.w3 = nn.Parameter(torch.tensor(-4.11))
        self.w4 = nn.Parameter(torch.tensor(2.74))

        self.loss = nn.MSELoss(reduction='sum')


    def forward(self, input_values):
        ## The forward() method is identical to what we used in Chapter 1.

        top_x_axis_values = (input_values * self.w1) + self.b1
        bottom_x_axis_values = (input_values * self.w2) + self.b2

        top_y_axis_values = F.relu(top_x_axis_values)
        bottom_y_axis_values = F.relu(bottom_x_axis_values)

        output_values = (top_y_axis_values * self.w3) + (bottom_y_axis_values * self.w4)

        return output_values


    def configure_optimizers(self): # this configures the optimizer we want to use for backpropagation.
        return SGD(self.parameters(), lr=0.01)


    def training_step(self, batch, batch_idx): # take a step during gradient descent.
        inputs, labels = batch # collect input
        outputs = self.forward(inputs) # run input through the neural network
        loss = self.loss(outputs, labels) 
        return loss

> `myNN` 클래스의 인스턴스를 만들고, `model.named_parameters()`로 현재 모델이 가지고 있는 모든 학습 가능한 파라미터(w1, b1, w2, b2, w3, w4)의 이름과 값을 출력합니다. 아직 학습을 하지 않았으므로, `__init__()`에서 지정한 무작위 초기값이 그대로 출력됩니다.

In [ ]:
model = myNN() # First, make model from the class

for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

> 학습 데이터의 입력값(`training_inputs`: 0.0, 0.5, 1.0)을 아직 학습되지 않은 모델에 통과시켜 출력값을 계산합니다. 원래 기대하는 정답은 `[0.0, 1.0, 0.0]`이지만, 모델이 아직 학습 전이라 전혀 다른(엉뚱한) 값이 나올 것입니다.

In [ ]:
## now run different doses through the neural network.
output_values = model(training_inputs)
torch.round(output_values, decimals=2)

> 그래프를 부드럽게 그리기 위해, `torch.linspace(start=0, end=1, steps=11)`로 0부터 1까지 균등하게 11개의 촘촘한 입력값(dose)을 만듭니다.

In [ ]:
input_doses = torch.linspace(start=0, end=1, steps=11)
input_doses

> 방금 만든 `input_doses`(0~1 사이 11개 값)를 (아직 학습되지 않은) 모델에 통과시켜 각 지점에서의 출력값을 계산합니다. 이 값들이 모델이 현재 그리는 곡선의 y축 좌표가 됩니다.

In [ ]:
output_values = model(input_doses)
output_values

> 학습 전 모델이 예측하는 곡선과, 실제 학습 데이터를 같은 그래프 위에 그려서 비교합니다.
>
> - 초록색(`color='green'`) 점과 선: `input_doses`에 대한 모델의 예측 곡선(`output_values`)입니다.
> - 주황색(`color='orange'`) 점: 실제 학습 데이터(`training_inputs`, `training_labels`) 3개 점입니다.
> - `.detach()`는 텐서에서 그래디언트(gradient) 추적 기능을 떼어내는 함수입니다. `output_values`는 모델의 파라미터로부터 계산된 값이라 역전파를 위한 계산 그래프(computation graph)가 연결되어 있는데, 그래프를 그릴 때는 이 연결이 필요 없으므로 `.detach()`로 떼어낸 뒤 `.numpy()`로 넘파이 배열로 변환해서 그립니다.
> - 아직 모델이 학습 전이므로, 초록색 곡선과 주황색 점들이 서로 크게 어긋나 있을 것으로 예상됩니다.

In [ ]:
## First, set the style for seaborn so that the graph looks cool.
sns.set(style="whitegrid")

## First, draw the individual output points
sns.scatterplot(x=input_doses,
                y=output_values.detach().numpy(),
                color='green',
                s=200)

## Now connect those points with a line
sns.lineplot(x=input_doses,
             y=output_values.detach().numpy(), ## NOTE: We call .detatch() because...
             color='green',
             linewidth=2.5)

## Add the values in the training dataset
sns.scatterplot(x=training_inputs,
                y=training_labels,
                color='orange',
                s=200)

## now label the y- and x-axes.
plt.ylabel('Effectiveness')
plt.xlabel('Dose')

### 신경망의 가중치(Weights)와 편향(Biases)을 실제로 학습

> 1. `L.Trainer()`로 **Lightning Trainer** 객체를 만듭니다. 이 객체가 학습 반복(epoch), 로깅, 체크포인트 저장 등 학습에 필요한 여러 설정을 관리합니다.
> 2. 그 Trainer의 `fit()` 메서드에 모델과 학습 데이터를 넘겨서, 실제 학습(역전파 + 파라미터 업데이트 반복)을 수행합니다.

> 실제로 모델을 학습시키는 코드입니다.
>
> - `model = myNN()` — 무작위 초기값을 가진 새로운 모델을 하나 만듭니다(이전에 만들었던 모델을 재사용하지 않고 깨끗한 상태에서 다시 시작합니다).
> - `trainer = L.Trainer(max_epochs=500, ...)` — 학습을 관리할 Trainer 객체를 만듭니다.
>   - `max_epochs=500` — 전체 학습 데이터를 최대 500번 반복해서 학습합니다.
>   - `logger=False` — 학습 로그를 별도로 기록하지 않습니다.
>   - `enable_checkpointing=False` — 학습 중간에 모델 체크포인트(저장 파일)를 만들지 않습니다.
>   - `enable_progress_bar=False` — 학습 진행률 표시줄(progress bar)을 표시하지 않습니다.
>   - (이 설정들은 이 튜토리얼처럼 간단한 예제에서 불필요한 출력/파일 생성을 줄이기 위한 것입니다.)
> - `trainer.fit(model, train_dataloaders=dataloader)` — 실제 학습을 시작합니다. 내부적으로 `dataloader`에서 배치를 꺼내 `training_step()`을 반복 호출하고, 계산된 손실(loss)을 바탕으로 역전파를 수행해 `w1~w4`, `b1`, `b2` 값을 조금씩 업데이트해 나갑니다.

In [ ]:
model = myNN()
## Now train the model...
trainer = L.Trainer(max_epochs=500, # how many times to go through the training data
                    logger=False,
                    enable_checkpointing=False,
                    enable_progress_bar=False)

trainer.fit(model, train_dataloaders=dataloader)

> 학습이 끝난 후, `model.named_parameters()`로 각 가중치·편향의 **새로운(학습된) 값**을 출력합니다.

In [ ]:
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=3))

> 학습이 완료된 모델에 `input_doses`(0~1 사이 11개 값)를 통과시켜 예측값을 계산하고, 소수점 둘째 자리로 반올림해서 출력합니다. 학습 전(셀 15)과 값을 비교해보면 차이를 확인할 수 있습니다.

In [ ]:
## now run the different doses through the neural network.
output_values = model(input_doses)
torch.round(output_values, decimals=2)

> 이제 학습이 끝난 모델이 예측하는 구부러진 모양(bent shape)을 다시 그래프로 그려서, 학습 데이터와 비교해볼 것이라는 설명입니다. 가중치·편향이 최적화되었으므로, 이론적으로는 곡선이 학습 데이터에 훨씬 더 잘 들어맞아야 합니다.

> 학습 후 모델의 예측 곡선(초록색)과 실제 학습 데이터(주황색 점)를 같은 그래프에 그립니다. 코드 구조는 학습 전 그래프(셀 16)와 완전히 동일하지만, 이번에는 `output_values`가 **학습된 모델**에서 나온 값이라는 점이 다릅니다. 학습이 잘 되었다면, 초록색 곡선이 주황색 점들 위를 거의 그대로 지나가는 것을 볼 수 있어야 합니다(즉, 모델의 예측과 실제 데이터가 거의 일치).

In [ ]:
## First, set the style for seaborn so that the graph looks cool.
sns.set(style="whitegrid")

## First, draw the individual output points
sns.scatterplot(x=input_doses,
                y=output_values.detach().numpy(),
                color='green',
                s=200)

## Now connect those points with a line
sns.lineplot(x=input_doses,
             y=output_values.detach().numpy(), ## NOTE: We call .detatch() because...
             color='green',
             linewidth=2.5)

## Add the values in the training dataset
sns.scatterplot(x=training_inputs,
                y=training_labels,
                color='orange',
                s=200)

## now label the y- and x-axes.
plt.ylabel('Effectiveness')
plt.xlabel('Dose')